# Patches MODIS e animação de NDVI — armazenamento local

Adaptação do [tutorial Google Earth Engine (2023, Apache-2.0)](https://github.com/google/earthengine-community/blob/master/guides/linked/Earth_Engine_training_patches_getPixels.ipynb).
O download usa `getPixels` e grava **um registro por ponto/ano** em TFRecord GZIP no disco local, sem bucket e sem autenticação do Colab.

Execute a partir de `src/lab/base`. Dependências: `earthengine-api`, `eerepr`, `pandas`, `numpy`, `matplotlib`, `tensorflow` e `ipython` no kernel Python. TensorFlow exige uma versão de Python compatível com sua distribuição.
Os helpers reutilizáveis ficam em `repository_ee.py`. O formato dos registros abaixo inclui metadados e difere do tutorial original.


In [ ]:
from pathlib import Path

import ee
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

from repository_ee import (
    PROJECT,
    AuthenticateEarhEngine,
    initialize_earth_engine,
    GeometryUtils,
    Modis,
    PeriodoAgricola,
    TrainingPatches,
)


## Autenticação e parâmetros

Use um projeto com Earth Engine habilitado e acesso pela conta autenticada. Comece com 2 pontos e 2025; para reproduzir o volume do tutorial, aumente para 64 pontos e `range(2010, 2023)`.
A grade padrão mede 2 × 2 km (8 × 8 pixels de 250 m), compatível com o buffer de raio 1000 m do exemplo. O CRS UTM 22S é apropriado para esta AOI no Tocantins; altere-o ao mudar de região.


In [ ]:

#initialize_earth_engine(project=PROJECT, use_eerepr=True)

ANOS = [2024,2025]
AMOSTRAS = 2
SEED = 42
ESCALA = 250
TAMANHO_PATCH = 8
CRS = "EPSG:32722"
MAX_WORKERS = 4
INDICE_REGISTRO = 1
SOBRESCREVER = False
SAIDA = Path("outputs/animation/modis-ndvi-v3.tfrecord.gz")

COORDS_TOCANTINS = [
    [-48.46995289774661, -10.859497023647087],
    [-47.84922535868411, -10.859497023647087],
    [-47.84922535868411, -10.362094203375317],
    [-48.46995289774661, -10.362094203375317],
    [-48.46995289774661, -10.859497023647087],
]
aoi = GeometryUtils.polygon(COORDS_TOCANTINS)


## Pontos e imagens

Transferimos apenas coordenadas e IDs pequenos com `getInfo()`, nunca coleções completas com pixels. A seed mantém os pontos reproduzíveis. Os IDs são consultados uma vez por ano, ordenados por data. `Modis.collection` usa o ano civil e expõe a banda `ndvi`; o download lê a banda original **NDVI** dos assets, pois `getPixels` não executa transformações da coleção.


In [ ]:
if not 1 <= AMOSTRAS <= 64:
    raise ValueError("Use de 1 a 64 pontos por execução.")
if not ANOS or len(set(ANOS)) != len(ANOS):
    raise ValueError("Informe anos distintos e não vazios.")

pontos = ee.FeatureCollection.randomPoints(
    region=aoi, points=AMOSTRAS, seed=SEED, maxError=1
).geometry().coordinates().getInfo()

grades = [
    TrainingPatches.grid(coords, TAMANHO_PATCH, ESCALA, CRS)
    for coords in pontos
]
ids_por_ano = {}
for ano in sorted(ANOS):
    periodo = PeriodoAgricola.tocantins(ano)
    modis = Modis.collection(aoi, period=periodo).sort("system:time_start")
    ids = modis.limit(32).aggregate_array("system:id").getInfo()
    if not ids:
        raise ValueError(f"Nenhuma imagem MODIS encontrada em {ano}.")
    ids_por_ano[ano] = ids

print(f"Registros: {len(pontos) * len(ANOS)}")
print(f"Requisições de pixels: {len(pontos) * sum(map(len, ids_por_ano.values()))}")
print("Imagens por ano:", {ano: len(ids) for ano, ids in ids_por_ano.items()})


## Download e gravação local

A concorrência é limitada e preserva a ordem temporal. Falhas interrompem a execução com o ID do asset; não descartamos quadros silenciosamente. O arquivo final só substitui o temporário após a conclusão. A memória contém apenas um ponto/ano por vez. Os valores MODIS são gravados sem reescala; a conversão para NDVI físico ocorre na visualização.


In [ ]:
if SAIDA.exists() and not SOBRESCREVER:
    raise FileExistsError(f"{SAIDA} já existe. Altere SAIDA ou SOBRESCREVER.")
SAIDA.parent.mkdir(parents=True, exist_ok=True)
temporario = SAIDA.with_name(SAIDA.name + ".partial")
if temporario.exists():
    raise FileExistsError(f"Remova ou renomeie o arquivo incompleto: {temporario}")

try:
    with tf.io.TFRecordWriter(str(temporario), options="GZIP") as writer:
        for indice_ponto, (coords, grade) in enumerate(zip(pontos, grades)):
            for ano, ids in ids_por_ano.items():
                patches = TrainingPatches.download(
                    ids, grade, band="NDVI", max_workers=MAX_WORKERS
                )
                example = TrainingPatches.to_example(patches, ids, coords, ano, grade)
                writer.write(example.SerializeToString())
                print(f"Ponto {indice_ponto + 1}/{len(pontos)}, {ano}: {patches.shape}")
    temporario.replace(SAIDA)
except Exception:
    temporario.unlink(missing_ok=True)
    raise

print("Arquivo salvo:", SAIDA.resolve())


## Inspeção de um registro

Lemos somente o registro escolhido, em vez de carregar todo o conjunto na memória. Os IDs e a grade permitem relacionar cada quadro ao asset e à posição espacial. Este exemplo não aplica filtro de qualidade por `SummaryQA`; os gráficos não devem ser interpretados como uma série livre de nuvens.


In [ ]:
if INDICE_REGISTRO < 0:
    raise ValueError("INDICE_REGISTRO deve ser não negativo.")
dataset = tf.data.TFRecordDataset(str(SAIDA), compression_type="GZIP")
registro = next(iter(dataset.skip(INDICE_REGISTRO).take(1)), None)
if registro is None:
    raise IndexError("INDICE_REGISTRO fora do arquivo.")
dados = TrainingPatches.parse_example(bytes(registro.numpy()))
print("Ano:", dados["year"], "Coordenadas:", dados["coordinates"])
print("Dimensões (tempo, altura, largura):", dados["patches"].shape)
print("Grade:", dados["grid"])


## Animação

NDVI MOD13Q1 tem fator 0,0001. Valores fora do intervalo válido bruto [-2000, 10000] são mascarados. Todos os quadros usam a mesma escala de cores e o primeiro também integra a animação. `to_jshtml()` permite exibir sem instalar FFmpeg.


In [ ]:
bruto = dados["patches"]
ndvi = np.ma.masked_outside(bruto, -2000, 10000) * 0.0001
nomes = [asset_id.rsplit("/", 1)[-1] for asset_id in dados["asset_ids"]]
fig, ax = plt.subplots(figsize=(6, 5))
imagem = ax.imshow(ndvi[0], vmin=-0.2, vmax=1.0, cmap="YlGn")
titulo = ax.set_title(nomes[0])
ax.set_xlabel("Coluna do patch")
ax.set_ylabel("Linha do patch")
fig.colorbar(imagem, ax=ax, label="NDVI")

def atualizar(indice: int):
    imagem.set_data(ndvi[indice])
    titulo.set_text(nomes[indice])
    return imagem, titulo

ani = animation.FuncAnimation(
    fig, atualizar, frames=len(ndvi), interval=150, blit=False, repeat_delay=1000
)
plt.close(fig)
display(HTML(ani.to_jshtml()))
